# Superset MCP — Creación y Gestión de Dashboards

Este notebook se conecta al MCP de Superset para:
1. Listar herramientas disponibles
2. Crear/actualizar dashboards
3. Consultar el estado del servidor

## Configuración
- MCP Server: `http://127.0.0.1:5011/mcp`

In [ ]:
import sys

sys.path.insert(0, '..')

from src.ts_superset.mcpUso import (
    call_tool,
    initialize_mcp,
    list_tools,
)

print("Modulos MCP importados.")

## 1. Inicializar sesion MCP

In [ ]:
print("Inicializando sesion MCP...")
init_result = initialize_mcp()
print(f"Estado: {init_result}")

## 2. Listar herramientas disponibles

In [ ]:
tools = list_tools()
print(f"Herramientas disponibles: {len(tools)}")
for t in tools:
    print(f"  - {t.get('name')}: {t.get('description', 'Sin descripcion')[:80]}")

## 3. Consultar estado de Superset

In [ ]:
result = call_tool("get_instance_info", {})
if result.get("content") and isinstance(result["content"], list):
    info = result["content"][0].get("text", str(result))
    try:
        import json
        print(json.dumps(json.loads(info), indent=2, ensure_ascii=False))
    except (json.JSONDecodeError, TypeError):
        print(info)
else:
    print(result)

## 4. Crear dashboard

Ajusta los parametros segun las herramientas disponibles en tu MCP.

In [ ]:
# Ejemplo: crear un dashboard
# Ajusta el nombre de la herramienta y los argumentos segun tu MCP

dashboard_config = {
    "dashboard_title": "Predicciones Crediticias Multi-Horizonte",
    "slug": "predicciones-crediticias",
    "published": True,
}

try:
    result = call_tool("create_dashboard", dashboard_config)
    print("Dashboard creado:")
    print(result)
except Exception as e:
    print(f"Error: {e}")
    print("Verifica el nombre de la herramienta con list_tools()")

## 5. Crear charts para el dashboard

Cada chart es una visualizacion que se agrega al dashboard.

In [ ]:
# Ejemplo: crear un chart de serie temporal de predicciones
# Ajusta segun las herramientas de tu MCP

chart_config = {
    "datasource_id": 1,  # ID del datasource en Superset
    "datasource_type": "table",
    "slice_name": "Evolucion Temporal de Predicciones",
    "viz_type": "line",
    "params": {
        "metrics": ["prob_media"],
        "groupby": ["mes"],
        "time_column": "mes",
    },
}

try:
    result = call_tool("create_chart", chart_config)
    print("Chart creado:")
    print(result)
except Exception as e:
    print(f"Error: {e}")

## 6. Agregar chart al dashboard

In [ ]:
# Ejemplo: agregar un chart a un dashboard existente
# Ajusta los IDs segun tus creaciones anteriores

add_config = {
    "dashboard_id": 1,  # ID del dashboard creado arriba
    "chart_id": 1,      # ID del chart creado arriba
    "position": 0,
}

try:
    result = call_tool("add_chart_to_dashboard", add_config)
    print("Chart agregado al dashboard:")
    print(result)
except Exception as e:
    print(f"Error: {e}")

## 7. Consultar dashboards existentes

In [ ]:
try:
    result = call_tool("list_dashboards", {})
    if result.get("content") and isinstance(result["content"], list):
        data = result["content"][0].get("text", str(result))
        try:
            import json
            dashboards = json.loads(data)
            if isinstance(dashboards, list):
                for d in dashboards:
                    print(f"  - ID: {d.get('id')}, Titulo: {d.get('dashboard_title', 'N/A')}")
            else:
                print(data)
        except (json.JSONDecodeError, TypeError):
            print(data)
    else:
        print(result)
except Exception as e:
    print(f"Error: {e}")

## 8. Ejecutar query SQL en Superset

In [ ]:
# Ejemplo: ejecutar una consulta SQL para verificar datos

sql_query = """
SELECT
    dt.mes,
    ROUND(AVG(fp.prob_media)::numeric, 4) AS prob_avg,
    SUM(fp.pred_media) AS n_crisis_pred
FROM fact_predicciones fp
JOIN dim_tiempo dt ON fp.id_tiempo = dt.id_tiempo
GROUP BY dt.mes
ORDER BY dt.mes
LIMIT 12
"""

try:
    result = call_tool("execute_sql", {"sql": sql_query})
    if result.get("content") and isinstance(result["content"], list):
        print(result["content"][0].get("text", str(result)))
    else:
        print(result)
except Exception as e:
    print(f"Error: {e}")

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*